In [59]:
import os

print(os.getcwd())

/Users/kreenasheth/anaconda_projects/94ca0db1-dbb5-4b6c-bb36-2abd43b71d43/ab testing sql


In [60]:
import pandas as pd

users = pd.read_csv("experiment_users.csv")
events = pd.read_csv("funnel_events.csv")

print(users.shape)
print(events.shape)

(30000, 13)
(81710, 6)


In [61]:
import duckdb

%load_ext sql
%sql duckdb:///:memory:

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## 1. Experiment Design

Control (A): Existing account-opening journey
Treatment (B): Redesigned account-opening journey

### Hypotheses
H₀: pB = pA — there is no difference in account-opening conversion.
H₁: pB > pA — the redesigned journey increases account-opening conversion.
A one-sided test is used because the business question specifically asks whether the redesign improves conversion.

### Primary KPI
Account-opening conversion rate
Users who opened an account / Total users in the experiment group

In [63]:
%%sql

-- Reset the existing DuckDB tables so the analysis starts from a clean state
DROP TABLE IF EXISTS experiment_users;
DROP TABLE IF EXISTS funnel_events;

Running query in 'duckdb:///:memory:'

Success


In [64]:
%%sql

-- Load the user-level experiment data into DuckDB
CREATE TABLE experiment_users AS
SELECT *
FROM read_csv_auto('experiment_users.csv');

Running query in 'duckdb:///:memory:'

Count


In [65]:
%%sql

-- Load the event-level funnel data into DuckDB
CREATE TABLE funnel_events AS
SELECT *
FROM read_csv_auto('funnel_events.csv');

Running query in 'duckdb:///:memory:'

Count


In [69]:
%%sql
Select * from experiment_users
limit 10

Running query in 'duckdb:///:memory:'

user_id,assigned_at,variant,channel,device,age_band,prior_investor,landing_page_view,started_application,completed_kyc,funded_account,account_opened,initial_assets_usd
100001,2026-08-26 16:25:10,B,Social,Desktop,25-34,0,1,1,1,0,0,0.0
100002,2026-08-26 19:11:24,A,Social,Mobile,55+,0,1,0,0,0,0,0.0
100003,2026-08-28 06:32:10,A,Partner,Desktop,25-34,0,1,1,1,1,1,9784.08
100004,2026-08-20 14:12:23,B,Social,Mobile,25-34,0,1,0,0,0,0,0.0
100005,2026-08-02 06:37:48,B,Paid Search,Mobile,35-44,0,1,1,1,1,1,7587.53
100006,2026-08-09 11:23:00,B,Direct,Desktop,25-34,0,1,1,1,1,1,3822.66
100007,2026-08-26 20:41:29,A,Organic Search,Mobile,25-34,0,1,1,1,0,0,0.0
100008,2026-08-19 04:06:46,B,Direct,Tablet,35-44,0,1,1,1,1,1,2938.81
100009,2026-08-24 04:27:54,B,Paid Search,Desktop,25-34,1,1,0,0,0,0,0.0
100010,2026-08-14 03:34:34,B,Organic Search,Mobile,35-44,0,1,0,0,0,0,0.0


In [70]:
%%sql
Select * from funnel_events
limit 10

Running query in 'duckdb:///:memory:'

user_id,event_name,event_time,variant,channel,device
100001,landing_page,2026-08-26 16:37:44,B,Social,Desktop
100001,application_start,2026-08-26 17:51:54,B,Social,Desktop
100001,kyc_complete,2026-08-26 17:45:40,B,Social,Desktop
100002,landing_page,2026-08-26 20:40:39,A,Social,Mobile
100003,landing_page,2026-08-28 07:55:46,A,Partner,Desktop
100003,application_start,2026-08-28 07:50:51,A,Partner,Desktop
100003,kyc_complete,2026-08-28 07:09:17,A,Partner,Desktop
100003,funding,2026-08-28 07:54:30,A,Partner,Desktop
100003,account_opened,2026-08-28 07:13:24,A,Partner,Desktop
100004,landing_page,2026-08-20 14:18:44,B,Social,Mobile


### 3. Data Validation & Statistical Assumptions
Before running the experiment analysis, validate the data and the assumptions relevant to a two-proportion Z-test.

#### Checks performed:
    
1. Total experiment population
2. Control/Treatment allocation
3. One row per user
4. Missing values in key fields
5. Binary outcome
6. Valid experiment variants


In [73]:
%%sql

-- Count the total number of users in the experiment
SELECT COUNT(*) AS total_users
FROM experiment_users;

Running query in 'duckdb:///:memory:'

total_users
30000


In [74]:
%%sql

-- Check how users are distributed between the Control (A) and Treatment (B) groups
SELECT
    variant,
    COUNT(DISTINCT user_id) AS users_count
FROM experiment_users
GROUP BY variant
ORDER BY variant;

Running query in 'duckdb:///:memory:'

variant,users_count
A,15052
B,14948


In [75]:
%%sql

-- Check whether any user appears more than once in the experiment data
SELECT
    user_id,
    COUNT(*) AS row_count
FROM experiment_users
GROUP BY user_id
HAVING COUNT(*) > 1;

Running query in 'duckdb:///:memory:'

user_id,row_count


In [76]:
%%sql

-- Check whether important experiment fields contain any missing values
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(user_id) AS missing_user_id,
    COUNT(*) - COUNT(variant) AS missing_variant,
    COUNT(*) - COUNT(account_opened) AS missing_account_opened
FROM experiment_users;

Running query in 'duckdb:///:memory:'

total_rows,missing_user_id,missing_variant,missing_account_opened
30000,0,0,0


In [77]:
%%sql

-- Check the possible values of the primary outcome to confirm it is binary
SELECT
    account_opened,
    COUNT(*) AS users
FROM experiment_users
GROUP BY account_opened
ORDER BY account_opened;

Running query in 'duckdb:///:memory:'

account_opened,users
0,21651
1,8349


In [78]:
%%sql

-- Check that the experiment contains only the expected Control (A) and Treatment (B) groups
SELECT
    variant,
    COUNT(DISTINCT user_id) AS users_count
FROM experiment_users
GROUP BY variant
ORDER BY variant;

Running query in 'duckdb:///:memory:'

variant,users_count
A,15052
B,14948


In [85]:
%%sql

-- Check whether the variant column contains any values other than the expected Control (A) and Treatment (B) groups

SELECT variant 
From experiment_users
where variant NOT IN ('A', 'B');

Running query in 'duckdb:///:memory:'

variant


### Step 3 — Calculate the Primary Metric

The primary metric is account-opening conversion.

The funnel is: Landing → Application Start → KYC → Funding → Account Opened


In [99]:
%%sql

-- Calculate the number of users at each stage of the account-opening funnel for the Control and Treatment groups
Select event_name, count(distinct user_id), variant
from funnel_events
group by event_name, variant
order by event_name, variant

Running query in 'duckdb:///:memory:'

event_name,count(DISTINCT user_id),variant
account_opened,3704,A
account_opened,4645,B
application_start,9285,A
application_start,10010,B
funding,4170,A
funding,5094,B
kyc_complete,6923,A
kyc_complete,7879,B
landing_page,15052,A
landing_page,14948,B


In [131]:
%%sql

-- Calculate total users, account-opened users, and account-opening conversion rate for the Control (A) and Treatment (B) groups
SELECT
    variant,
    COUNT(DISTINCT user_id) AS total_users,
    COUNT(DISTINCT CASE WHEN account_opened = 1 THEN user_id END) AS account_opened_users,
    ROUND(
        COUNT(DISTINCT CASE WHEN account_opened = 1 THEN user_id END) * 100.0
        / COUNT(DISTINCT user_id),
        2
    ) AS conversion_rate
FROM experiment_users
GROUP BY variant
ORDER BY variant;

Running query in 'duckdb:///:memory:'

variant,total_users,account_opened_users,conversion_rate
A,15052,3704,24.61
B,14948,4645,31.07


### Conversion Lift

Two views of lift are reported:

Absolute lift: Treatment conversion − Control conversion

Relative lift: Absolute lift / Control conversion

In [134]:
%%sql

-- Calculate the absolute and relative conversion lift of Treatment (B) compared with Control (A)
SELECT
    control_conversion,
    treatment_conversion,
    ROUND(treatment_conversion - control_conversion, 2) AS absolute_lift,
    ROUND(
        (treatment_conversion - control_conversion)
        / control_conversion * 100,
        2
    ) AS relative_lift
FROM (
    SELECT
        ROUND(
            AVG(CASE WHEN variant = 'A' THEN account_opened END) * 100,
            2
        ) AS control_conversion,
        ROUND(
            AVG(CASE WHEN variant = 'B' THEN account_opened END) * 100,
            2
        ) AS treatment_conversion
    FROM experiment_users
) AS conversion_rates;

Running query in 'duckdb:///:memory:'

control_conversion,treatment_conversion,absolute_lift,relative_lift
24.61,31.07,6.46,26.25


### Step 4 — Select & Conduct Statistical Test

The primary KPI is a binary outcome and the Control and Treatment groups are independent, so a two-proportion Z-test is used.

The test workflow is:

1. Calculate successes and sample sizes
2. Calculate the pooled proportion
3. Calculate the standard error
4. Calculate the Z-statistic
5. Calculate the one-sided p-value

In [138]:
%%sql

-- Calculate the number of account openings and total users for the Control (A) and Treatment (B) groups to prepare for the two-proportion Z-test
SELECT
    variant,
    COUNT(DISTINCT user_id) AS total_users,
    COUNT(DISTINCT CASE WHEN account_opened = 1 THEN user_id END) AS account_opened_users
FROM experiment_users
GROUP BY variant
ORDER BY variant;

Running query in 'duckdb:///:memory:'

variant,total_users,account_opened_users
A,15052,3704
B,14948,4645


In [139]:
%%sql

-- Calculate the pooled conversion proportion across the Control (A) and Treatment (B) groups for the two-proportion Z-test
SELECT
    SUM(account_opened) * 1.0 / COUNT(*) AS pooled_proportion
FROM experiment_users;

Running query in 'duckdb:///:memory:'

pooled_proportion
0.2783


In [142]:
%%sql
    
-- Calculate the Z-statistic using intermediate conversion and standard error calculations
WITH z_test_inputs AS (
    SELECT
        SUM(CASE WHEN variant = 'A' THEN account_opened ELSE 0 END) * 1.0
            / SUM(CASE WHEN variant = 'A' THEN 1 ELSE 0 END) AS control_conversion,

        SUM(CASE WHEN variant = 'B' THEN account_opened ELSE 0 END) * 1.0
            / SUM(CASE WHEN variant = 'B' THEN 1 ELSE 0 END) AS treatment_conversion,

        SUM(account_opened) * 1.0 / COUNT(*) AS pooled_proportion,

        SQRT(
            (SUM(account_opened) * 1.0 / COUNT(*))
            * (1 - SUM(account_opened) * 1.0 / COUNT(*))
            * (
                1.0 / SUM(CASE WHEN variant = 'A' THEN 1 ELSE 0 END)
                + 1.0 / SUM(CASE WHEN variant = 'B' THEN 1 ELSE 0 END)
            )
        ) AS standard_error

    FROM experiment_users
)

SELECT
    control_conversion,
    treatment_conversion,
    pooled_proportion,
    standard_error,
    (treatment_conversion - control_conversion) / standard_error AS z_statistic
FROM z_test_inputs;

Running query in 'duckdb:///:memory:'

control_conversion,treatment_conversion,pooled_proportion,standard_error,z_statistic
0.24608025511559925,0.3107439122290607,0.2783,0.005174959243497855,12.495491089076491


In [145]:
%%sql

-- Load statistical distribution functions required to calculate the p-value
INSTALL stats_duck FROM community;
LOAD stats_duck;

Running query in 'duckdb:///:memory:'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success


In [146]:
%%sql

-- Calculate the one-sided p-value from the Z-statistic
WITH z_test_inputs AS (
    SELECT
        SUM(CASE WHEN variant = 'A' THEN account_opened ELSE 0 END) * 1.0
            / SUM(CASE WHEN variant = 'A' THEN 1 ELSE 0 END) AS control_conversion,

        SUM(CASE WHEN variant = 'B' THEN account_opened ELSE 0 END) * 1.0
            / SUM(CASE WHEN variant = 'B' THEN 1 ELSE 0 END) AS treatment_conversion,

        SQRT(
            (SUM(account_opened) * 1.0 / COUNT(*))
            * (1 - SUM(account_opened) * 1.0 / COUNT(*))
            * (
                1.0 / SUM(CASE WHEN variant = 'A' THEN 1 ELSE 0 END)
                + 1.0 / SUM(CASE WHEN variant = 'B' THEN 1 ELSE 0 END)
            )
        ) AS standard_error
    FROM experiment_users
),

z_score AS (
    SELECT
        (treatment_conversion - control_conversion) / standard_error AS z_statistic
    FROM z_test_inputs
)

SELECT
    z_statistic,
    1 - pnorm(z_statistic) AS p_value
FROM z_score;

Running query in 'duckdb:///:memory:'

z_statistic,p_value
12.495491089076491,0.0


### Step 5 — Confidence Interval

A 95% confidence interval is calculated for the difference in conversion rates:

Treatment (B) − Control (A)

This provides a range for the estimated treatment effect and helps quantify the uncertainty around the observed lift

In [147]:
%%sql

-- Calculate the 95% confidence interval for the difference in conversion rates between Treatment (B) and Control (A)
WITH conversion_stats AS (
    SELECT
        SUM(CASE WHEN variant = 'A' THEN account_opened ELSE 0 END) * 1.0
            / SUM(CASE WHEN variant = 'A' THEN 1 ELSE 0 END) AS control_conversion,

        SUM(CASE WHEN variant = 'B' THEN account_opened ELSE 0 END) * 1.0
            / SUM(CASE WHEN variant = 'B' THEN 1 ELSE 0 END) AS treatment_conversion,

        SUM(CASE WHEN variant = 'A' THEN 1 ELSE 0 END) AS control_users,

        SUM(CASE WHEN variant = 'B' THEN 1 ELSE 0 END) AS treatment_users

    FROM experiment_users
),

ci_stats AS (
    SELECT
        treatment_conversion - control_conversion AS conversion_difference,

        SQRT(
            (control_conversion * (1 - control_conversion) / control_users)
            +
            (treatment_conversion * (1 - treatment_conversion) / treatment_users)
        ) AS standard_error
    FROM conversion_stats
)

SELECT
    conversion_difference,
    conversion_difference - 1.96 * standard_error AS ci_lower,
    conversion_difference + 1.96 * standard_error AS ci_upper
FROM ci_stats;

Running query in 'duckdb:///:memory:'

conversion_difference,ci_lower,ci_upper
0.06466365711346148,0.054544651505707906,0.07478266272121505


### Business Findings

Experiment result
1. Control conversion: 24.61%
2. Treatment conversion: 31.07%
3. Absolute lift: +6.46 percentage points
4. Relative lift: +26.25%
5. Z-statistic: 12.49
6. p-value: < 0.001
7. 95% CI for the conversion difference: 5.45 to 7.48 percentage points

Interpretation

Treatment B shows a higher account-opening conversion rate than Control A. The confidence interval remains above zero, and the statistical test provides strong evidence against the null hypothesis.

From a business perspective, the observed improvement is large enough to warrant further product investigation rather than being treated as a statistically detectable but trivial change.

In a real experiment, the next checks would include segment performance, guardrail metrics, experiment duration, and implementation cost before making a rollout decision.